# Model comparison: learned dispersion (theta) distributions

Load trained models and compare the learned per-gene inverse dispersion parameter $\theta = \exp(\texttt{px\_r})$.

- **scvi_default**: standard scVI, all defaults (ZINB, n_hidden=128, n_latent=10), no dispersion prior
- **scvi_custom**: standard scVI, large arch (NB, n_hidden=512, n_latent=128), no dispersion prior
- **scvi_custom_default_train**: standard scVI, large arch (NB), default training (172 epochs)
- **regularizedvi_gamma_poisson**: GammaPoisson mode, prior on $1/\sqrt{\theta}$ (pushes $\theta$ large, toward Poisson)
- **regularizedvi_nb** (tutorial): NB mode, prior on $\sqrt{\theta}$ (pushes $\theta$ small)

The Exponential(rate=3) prior has:
- GammaPoisson direction: $E[1/\sqrt{\theta}] = 1/3$ → $\theta$ centered ~9
- NB direction: $E[\sqrt{\theta}] = 1/3$ → $\theta$ centered ~1/9

Key question: do the unconstrained scVI models learn theta values that are very different from both 1/9 and 9?

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from pathlib import Path

rcParams["pdf.fonttype"] = 42
rcParams["figure.figsize"] = 12, 5

## 1. Load px_r from saved models

Extract the unconstrained dispersion parameter `px_r` from each saved model checkpoint.

In [ ]:
results_dir = Path("results")

# Models to load: (display_name, folder_name, description)
model_specs = [
    ("scvi_default", "scvi_default", "ZINB, 128/10, default train, no prior"),
    ("scvi_custom", "scvi_custom", "NB, 512/128, custom train, no prior"),
    ("scvi_custom_default_train", "scvi_custom_default_train", "NB, 512/128, default train, no prior"),
    ("scvi_custom_zinb", "scvi_custom_zinb", "ZINB, 512/128, custom train, no prior"),
    ("regularizedvi_gamma_poisson", "regularizedvi_gamma_poisson", "GammaPoisson, 512/128, prior→Poisson"),
    ("regularizedvi_nb", "regularizedvi_nb", "NB, 512/128, prior→overdispersed"),
    ("regularizedvi_gp_small", "regularizedvi_gamma_poisson_small", "GammaPoisson, 128/10, prior→Poisson"),
    ("regularizedvi_gp_medium", "regularizedvi_gamma_poisson_medium", "GammaPoisson, 256/30, prior→Poisson"),
    (
        "regularizedvi_gp_small_default",
        "regularizedvi_gamma_poisson_small_default_train",
        "GammaPoisson, 128/10, default train, prior→Poisson",
    ),
]

models = {}
for name, folder, desc in model_specs:
    model_path = results_dir / folder / "model" / "model.pt"
    if not model_path.exists():
        print(f"SKIP {name}: {model_path} not found")
        continue
    state = torch.load(model_path, map_location="cpu", weights_only=False)
    px_r = state["model_state_dict"]["px_r"]
    theta = torch.exp(px_r).detach().numpy()
    # For gene-batch dispersion (2D), take median across batches
    if theta.ndim == 2:
        theta_per_gene = np.median(theta, axis=1)
    else:
        theta_per_gene = theta
    models[name] = {"theta": theta_per_gene, "theta_full": theta, "desc": desc}
    print(
        f"{name}: shape={px_r.shape}, theta range=[{theta_per_gene.min():.2f}, {theta_per_gene.max():.2f}], "
        f"median={np.median(theta_per_gene):.2f}, mean={theta_per_gene.mean():.2f}"
    )

## 2. Theta distributions (log scale)

Histogram of $\log_{10}(\theta)$ for each model. Reference lines at $\theta = 1/9$ (NB prior center) and $\theta = 9$ (GammaPoisson prior center).

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 4), sharey=True)
if len(models) == 1:
    axes = [axes]

bins = np.linspace(-3, 3, 80)  # log10 scale: 0.001 to 1000

for ax, (name, data) in zip(axes, models.items(), strict=False):
    log_theta = np.log10(data["theta"])
    ax.hist(log_theta, bins=bins, alpha=0.7, edgecolor="black", linewidth=0.3)
    ax.axvline(np.log10(1 / 9), color="red", linestyle="--", alpha=0.7, label="θ=1/9 (NB prior)")
    ax.axvline(np.log10(9), color="blue", linestyle="--", alpha=0.7, label="θ=9 (GP prior)")
    ax.axvline(
        np.log10(np.median(data["theta"])),
        color="green",
        linestyle="-",
        alpha=0.9,
        label=f"median={np.median(data['theta']):.1f}",
    )
    ax.set_xlabel("log₁₀(θ)")
    ax.set_title(name, fontsize=9)
    ax.legend(fontsize=7)

axes[0].set_ylabel("Gene count")
fig.suptitle("Per-gene inverse dispersion θ = exp(px_r)", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Pairwise theta comparison (scatter)

Scatter plot of $\log_{10}(\theta)$ between pairs of models to see gene-level correlations.

In [ ]:
# Compare each model against scvi_default as baseline (if available)
baseline = "scvi_default"
if baseline in models:
    others = [k for k in models if k != baseline]
    n = len(others)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]

    for ax, other_name in zip(axes, others, strict=False):
        x = np.log10(models[baseline]["theta"])
        y = np.log10(models[other_name]["theta"])
        # Ensure same number of genes
        n_genes = min(len(x), len(y))
        ax.hist2d(x[:n_genes], y[:n_genes], bins=80, cmap="viridis", range=[[-3, 3], [-3, 3]])
        ax.plot([-3, 3], [-3, 3], "r--", alpha=0.5)
        ax.set_xlabel(f"log₁₀(θ) — {baseline}")
        ax.set_ylabel(f"log₁₀(θ) — {other_name}")
        ax.set_title(other_name, fontsize=9)

    fig.suptitle(f"Per-gene θ: each model vs {baseline}", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print(f"Baseline model '{baseline}' not found")

## 4. Implied variance inflation

NB variance = $\mu + \mu^2/\theta$. The variance inflation factor (beyond Poisson) is $\mu/\theta$. Compare this across models for typical expression levels.

In [ ]:
mu_typical = 5.0  # typical gene expression

print(f"Variance = μ + μ²/θ = {mu_typical} + {mu_typical**2}/θ")
print(f"{'Model':<35} {'Median θ':>10} {'Var(x)':>10} {'Var/μ':>10} {'Poisson Var':>12}")
print("-" * 80)
for name, data in models.items():
    med_theta = np.median(data["theta"])
    var = mu_typical + mu_typical**2 / med_theta
    print(f"{name:<35} {med_theta:>10.2f} {var:>10.2f} {var / mu_typical:>10.2f} {mu_typical:>12.2f}")

## 5. Per-batch theta variation (regularizedvi only)

For models with `dispersion="gene-batch"`, check how much theta varies across batches for each gene.

In [ ]:
for name, data in models.items():
    theta_full = data["theta_full"]
    if theta_full.ndim == 2:
        cv = theta_full.std(axis=1) / theta_full.mean(axis=1)  # coefficient of variation
        print(f"{name}: gene-batch theta, shape={theta_full.shape}")
        print(f"  CV across batches: median={np.median(cv):.3f}, mean={cv.mean():.3f}, max={cv.max():.3f}")

        plt.hist(cv, bins=50, alpha=0.7, edgecolor="black", linewidth=0.3)
        plt.xlabel("CV of θ across batches")
        plt.ylabel("Gene count")
        plt.title(f"{name}: per-gene variation of θ across {theta_full.shape[1]} batches")
        plt.show()

## Summary

| Model | Prior | Median θ | Interpretation |
|-------|-------|----------|----------------|
| scvi_default | None | ~1.3 | Moderate overdispersion |
| scvi_custom | None | ~2.4 | Moderate overdispersion |
| regularizedvi_nb | Exp on √θ (→small) | ? | θ→1/9 expected |
| regularizedvi_gamma_poisson | Exp on 1/√θ (→large) | ~53 | Near Poisson |

Both prior directions constrain θ via √ transform, but the unconstrained models land at θ~1-2, which is far from both 1/9 and 9. This suggests the prior's main effect is constraining θ to a definite range rather than the specific direction mattering.